# PepSurf_G ∪ EpiSearch - Analysis Workflow

**A Google Colab notebook for integrated epitope prediction and analysis**

This notebook performs a comprehensive workflow combining:
- **PepSurf**: Extracts and processes epitope information
- **Greedy Algorithm**: Identifies minimal peptide sets for epitope coverage
- **EpiSearch**: Predicts epitope residues from peptide sequences
- **PepSurf_G ∪ EpiSearch**: Determine the epitope from the union of PepSurf and EpiSearch results

## Workflow Steps
1. Run PepSurf to get the input files and upload them
2. Set the antigen name for the current analysis (used for IDs and output files)
3. Run PepSurf processing + greedy minimal peptide selection (PepSurf_G)
4. Upload the EpiSearch input file and combine PepSurf + EpiSearch outputs
5. Generate the union epitope table and download final results
6. Upload the antigen 3D structure and visualize the predicted epitope


All scripts are embedded in this notebook. You can simply run it on your data files.

To run the analyzes:
- Download this Notebook file (.ipynb)
- Access: https://colab.research.google.com/
- Upload the .ipynb file
- Run the analysis step-by-step

## PepSurf analysis
Before running Step 1, you need to run PepSurf analysis to get the following outputs: 
`pepSurfServer.res`
`*_significantPaths.txt`

You can download the required softwares (windows version) below:

- Surface Racer: obtained at https://tsodikovlab.createuky.net/index_files/Surface_Racer.htm
- PepSurf: obtained at http://pepitope.tau.ac.il/
- Surface Racer/PepSurf softwares pack (zip): https://github.com/aldemobr/PepSurf_G-EpiSearch---Analysis-Workflow/blob/cb5f4f431c4d557158e5dc38138ef1783382fb4d/PepSurf_softwares_download.zip

Run the PepSurf locally before you proceed with this notebook.

In [ ]:
# Step 1: Upload input files
from google.colab import files

print("Upload pepSurfServer.res and all *_significantPaths.txt files")
uploaded = files.upload()

# The files are now in the current working directory

In [ ]:
# Step 2: Set antigen name for this run and press Enter
antigen_name = input("Enter antigen name (used for output files and peptide IDs): ").strip()
if not antigen_name:
    raise ValueError("Antigen name cannot be empty")

print(f"Antigen set to: {antigen_name}")

In [ ]:
#@title PepSurf_G scripts (click to expand)
# Step 3: Embedded PepSurf processing + greedy minimal set scripts
import os
import re
import csv
from collections import OrderedDict

def process_pepsurf_file(input_file, antigen_name):
    # Uses antigen_name from the input cell, not the directory name
    with open(input_file, "r") as f:
        content = f.read()

    score_match = re.search(r"Score:\s*([\d.]+)", content)
    if not score_match:
        raise ValueError("Score not found in pepSurfServer.res")
    score = score_match.group(1)

    residues = re.findall(r"^([A-Z]{3}\d+)", content, re.MULTILINE)

    peptides = []
    peptide_section = re.search(
        r"Peptides participating in this cluster:\nID\tSequence\n(.*?)\+{10}",
        content,
        re.DOTALL,
    )
    if peptide_section:
        for line in peptide_section.group(1).strip().split("\n"):
            if line.strip():
                parts = line.split("\t")
                if len(parts) == 2:
                    peptide_id = f"{antigen_name}_{int(parts[0]) - 1}"
                    sequence = parts[1].strip()
                    peptides.append((peptide_id, sequence))

    output_lines = [
        "Antigen\tNumber of Clusters\tScore\tAmino acids\tAll_peptides_in_cluster (ID)\tAll_peptides_in_cluster (Sequence)",
        f'{antigen_name}\t1\t{score}\t"{"; ".join(residues)}"',
    ]

    for pid, seq in peptides:
        output_lines.append(f"\t\t\t\t{pid}\t{seq}")

    output_file = f"{antigen_name}.txt"
    with open(output_file, "w") as f:
        f.write("\n".join(output_lines) + "\n")

    return output_file

def extract_path_residues(file_path):
    residues = OrderedDict()
    try:
        with open(file_path, "r") as f:
            content = f.read()

        for block in content.split("_________________________________"):
            p_value_match = re.search(r"P-value:\s*([\d.e-]+)", block)
            if not p_value_match:
                continue
            if float(p_value_match.group(1)) > 0.05:
                continue

            path_match = re.search(r"path:\s*([^\n]+)", block)
            if path_match:
                for residue in path_match.group(1).strip().split():
                    residues[residue] = None
    except FileNotFoundError:
        pass

    return " ".join(residues.keys())

def process_peptides(input_file):
    base_name = os.path.splitext(input_file)[0]
    output_file = f"{base_name}_w_paths.csv"

    with open(input_file, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    headers = [
        "Antigen",
        "Number_of_the_Cluster",
        "PepSurf_Score",
        "Amino_acids",
        "Peptide_ID",
        "Peptide_Sequence",
        "Path_Residues",
    ]
    output_lines = [",".join(headers)]

    current_antigen = None
    for line in lines[1:]:
        parts = [p.strip() for p in line.split("\t") if p.strip()]

        if len(parts) >= 3 and parts[0]:
            current_antigen = parts[0]
            output_lines.append(
                f"{current_antigen},"
                f"{parts[1] if len(parts) > 1 else ''},"
                f"{parts[2] if len(parts) > 2 else ''},"
                f'"{parts[3] if len(parts) > 3 else ""}",,',
            )
        elif len(parts) >= 2:
            peptide_id = parts[-2]
            peptide_sequence = parts[-1]
            peptide_num = peptide_id.split("_")[-1]
            sig_path_file = f"{peptide_num}_significantPaths.txt"
            path_residues = extract_path_residues(sig_path_file)

            output_lines.append(
                f"{current_antigen},,,,"
                f"{peptide_id},"
                f"{peptide_sequence},"
                f'"{path_residues}"',
            )

    with open(output_file, "w", encoding="utf-8-sig") as f:
        f.write("\n".join(output_lines))

    return output_file

def extract_position(aa):
    return int("".join(filter(str.isdigit, aa)))

def parse_amino_acids(aa_string):
    cleaned = aa_string.replace('""', '"').strip('"')
    return [aa.strip() for aa in cleaned.split(";") if aa.strip()]

def clean_residue(residue):
    if residue.endswith("A") and len(residue) > 1:
        code_map = {
            "A": "ALA", "C": "CYS", "D": "ASP", "E": "GLU",
            "F": "PHE", "G": "GLY", "H": "HIS", "I": "ILE",
            "K": "LYS", "L": "LEU", "M": "MET", "N": "ASN",
            "P": "PRO", "Q": "GLN", "R": "ARG", "S": "SER",
            "T": "THR", "V": "VAL", "W": "TRP", "Y": "TYR",
        }
        aa_code = code_map.get(residue[0], residue[0])
        position = residue[1:-1]
        return f"{aa_code}{position}"
    return residue

def read_csv_file(filename):
    peptides = {}
    antigen_info = {}
    all_amino_acids_in_antigen = set()

    with open(filename, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("Amino_acids", "").strip():
                antigen_info = {
                    "name": row.get("Antigen", "").strip(),
                    "amino_acids": parse_amino_acids(row["Amino_acids"]),
                    "score": row.get("PepSurf_Score", "").strip(),
                    "num_clusters": row.get("Number_of_the_Cluster", "").strip(),
                }
                all_amino_acids_in_antigen.update(antigen_info["amino_acids"])
            elif row.get("Peptide_ID", "").strip() and row.get("Peptide_Sequence", "").strip():
                peptide_id = row["Peptide_ID"].strip()
                path_residues = [clean_residue(r) for r in row.get("Path_Residues", "").split()]

                peptides[peptide_id] = {
                    "sequence": row["Peptide_Sequence"].strip(),
                    "path_residues": path_residues,
                    "amino_acids_covered": set(),
                    "is_required": False,
                }

                for residue in path_residues:
                    for aa in all_amino_acids_in_antigen:
                        if aa in residue:
                            peptides[peptide_id]["amino_acids_covered"].add(aa)

    all_covered = set()
    for p in peptides.values():
        all_covered.update(p["amino_acids_covered"])
    target_amino_acids = sorted(all_covered, key=extract_position)
    total_coverable = len(target_amino_acids)

    for pid in peptides:
        covered = peptides[pid]["amino_acids_covered"]
        peptides[pid]["coverage_n"] = len(covered)
        peptides[pid]["coverage_pct"] = (len(covered) / total_coverable) * 100 if total_coverable > 0 else 0

    return antigen_info, peptides, target_amino_acids, total_coverable

def greedy_set_cover(peptides, target_amino_acids):
    target_set = set(target_amino_acids)
    covered = set()
    selected_peptides = []

    while covered != target_set:
        best_peptide = None
        best_new = set()

        for pid, data in peptides.items():
            if pid in selected_peptides:
                continue
            new_covered = data["amino_acids_covered"] - covered
            if len(new_covered) > len(best_new):
                best_new = new_covered
                best_peptide = pid

        if not best_peptide:
            break

        covered.update(peptides[best_peptide]["amino_acids_covered"])
        selected_peptides.append(best_peptide)

    for pid in peptides:
        peptides[pid]["is_required"] = pid in selected_peptides

    return selected_peptides

def generate_output_files(base_name, antigen_info, peptides, target_amino_acids, total_coverable, minimal_set):
    csv_output = f"{base_name}_minimal_peptides.csv"
    with open(csv_output, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "Antigen", "Number_of_the_Cluster", "PepSurf_Score", "Amino_acids",
            "Peptide_ID", "Peptide_Sequence", "Path_Residues", "Minimal_peptide",
            "Coverage_n", "Coverage_%",
        ])

        first_peptide = True
        for pid, data in peptides.items():
            coverage_str = f"{data['coverage_n']}/{total_coverable}"
            coverage_pct = f"{data['coverage_pct']:.2f}%"

            if first_peptide:
                writer.writerow([
                    antigen_info.get("name", ""),
                    antigen_info.get("num_clusters", ""),
                    antigen_info.get("score", ""),
                    "; ".join(antigen_info.get("amino_acids", [])),
                    pid,
                    data["sequence"],
                    " ".join(data["path_residues"]),
                    "Required" if data["is_required"] else "Non-required",
                    coverage_str,
                    coverage_pct,
                ])
                first_peptide = False
            else:
                writer.writerow([
                    "", "", "", "",
                    pid,
                    data["sequence"],
                    " ".join(data["path_residues"]),
                    "Required" if data["is_required"] else "Non-required",
                    coverage_str,
                    coverage_pct,
                ])

    txt_output = f"{base_name}_minimal_peptides_details.txt"
    with open(txt_output, "w", encoding="utf-8") as f:
        f.write(f"Antigen: {antigen_info.get('name', '')}\n")
        f.write(f"Score: {antigen_info.get('score', '')}\n")
        f.write(f"Number of the Cluster: {antigen_info.get('num_clusters', '')}\n\n")

        f.write(f"Total amino acids in the epitope: {len(antigen_info.get('amino_acids', []))}\n")
        f.write(f"Total coverable amino acids by peptides: {total_coverable}\n")
        if len(antigen_info.get('amino_acids', [])) > 0:
            pct = (total_coverable / len(antigen_info.get('amino_acids', []))) * 100
        else:
            pct = 0
        f.write(f"Coverage percentage: {pct:.2f}%\n\n")

        f.write(f"Target amino acids to cover ({total_coverable}):\n")
        f.write(", ".join(target_amino_acids) + "\n\n")

        f.write(f"Total peptides available: {len(peptides)}\n")

        required_peptides = [p for p in peptides.values() if p["is_required"]]
        non_required = [p for p in peptides.values() if not p["is_required"]]

        f.write(f"\nMinimal peptide set required ({len(required_peptides)} peptides):\n")
        for data in required_peptides:
            f.write(f"{data['sequence']}\t({data['coverage_n']}/{total_coverable}; {data['coverage_pct']:.2f}%)\t")
            f.write(", ".join(sorted(data["amino_acids_covered"], key=extract_position)) + "\n")

        if non_required:
            f.write(f"\nNon-required peptides ({len(non_required)} peptides):\n")
            for data in non_required:
                f.write(f"{data['sequence']}\t({data['coverage_n']}/{total_coverable}; {data['coverage_pct']:.2f}%)\t")
                f.write(", ".join(sorted(data["amino_acids_covered"], key=extract_position)) + "\n")

        uncovered = set(antigen_info.get("amino_acids", [])) - set(target_amino_acids)
        if uncovered:
            f.write(f"\nUncovered amino acids in the epitope ({len(uncovered)}):\n")
            f.write(", ".join(sorted(uncovered, key=extract_position)) + "\n")

        covered_by_minimal = set()
        for pid in minimal_set:
            covered_by_minimal.update(peptides[pid]["amino_acids_covered"])
        coverage = len(covered_by_minimal) / total_coverable * 100 if total_coverable > 0 else 0
        f.write(f"\nMinimal set coverage: {coverage:.2f}% ({len(covered_by_minimal)}/{total_coverable})\n")

    return csv_output, txt_output

def run_pepsurf_greedy_pipeline(antigen_name):
    input_file = "pepSurfServer.res"
    if not os.path.exists(input_file):
        raise FileNotFoundError("pepSurfServer.res not found")

    step1_output = process_pepsurf_file(input_file, antigen_name)
    step2_output = process_peptides(step1_output)

    antigen_info, peptides, target_amino_acids, total_coverable = read_csv_file(step2_output)
    minimal_set = greedy_set_cover(peptides, target_amino_acids)
    step3_csv, step3_txt = generate_output_files(
        os.path.splitext(step2_output)[0].replace("_w_paths", ""),
        antigen_info,
        peptides,
        target_amino_acids,
        total_coverable,
        minimal_set,
    )

    return step1_output, step2_output, step3_csv, step3_txt

In [ ]:
# Step 4: Run PepSurf + greedy pipeline
step1_output, step2_output, step3_csv, step3_txt = run_pepsurf_greedy_pipeline(antigen_name)

print("Generated files:")
print("-", step1_output)
print("-", step2_output)
print("-", step3_csv)
print("-", step3_txt)

**Expected output:** Printed file list and generated `<antigen_name>.txt`, `<antigen_name>_w_paths.csv`, `<antigen_name>_minimal_peptides.csv`, `<antigen_name>_minimal_peptides_details.txt`.

In [ ]:
# Step 4b: Download PepSurf + greedy results
from google.colab import files
import zipfile

output_files = [step3_csv, step3_txt]
missing = [f for f in output_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing outputs: {missing}")

zip_name = f"{antigen_name}_pepsurf_greedy_outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in output_files:
        zf.write(fpath)

files.download(zip_name)

**Expected output:** Browser downloads `<antigen_name>_pepsurf_greedy_outputs.zip`.

## EpiSearch analysis
After running Step 4, collect the peptides classified as **Required** in the PepSurf output and run them in EpiSearch (use their new webserver at https://curie.sdaponline.org/episearch.html).

Save the best EpiSearch solution (with the highest Score) by simply copying it (from "Solution No..." to "Predicted length::XX") into a text file to be named:

`<antigen_name>_episearch.txt`

Then upload that file below.

In [ ]:
# Step 5: Upload EpiSearch best-solution file
from google.colab import files

expected_episearch = f"{antigen_name}_episearch.txt"
print(f"Upload the file named: {expected_episearch}")
episearch_upload = files.upload()

In [ ]:
#@title EpiSearch + union scripts (click to expand)
# Step 6: Embedded EpiSearch processing + union scripts (steps 4-6)
import glob
from pathlib import Path
import pandas as pd

AMINO_ACID_CODES = {
    "A": "ALA", "R": "ARG", "N": "ASN", "D": "ASP", "C": "CYS",
    "Q": "GLN", "E": "GLU", "G": "GLY", "H": "HIS", "I": "ILE",
    "L": "LEU", "K": "LYS", "M": "MET", "F": "PHE", "P": "PRO",
    "S": "SER", "T": "THR", "W": "TRP", "Y": "TYR", "V": "VAL",
}

def convert_to_three_letter_code(residue_list):
    converted_list = []
    for residue in residue_list:
        if not residue or len(residue) < 2:
            continue
        one_letter_code = residue[0]
        number = residue[1:]
        three_letter_code = AMINO_ACID_CODES.get(one_letter_code.upper(), one_letter_code)
        converted_list.append(f"{three_letter_code}{number}")
    return converted_list

def process_file_content(raw_text, protein_name):
    file_info_match = re.search(
        r"Center Residue\s*::\s*(\d+)\s*Score:\s*([\d.]+)\s*\[([\d.]+)\]",
        raw_text,
        re.IGNORECASE,
    )

    center_residue, total_score, avg_score = ("", "", "")
    if file_info_match:
        center_residue = file_info_match.group(1)
        total_score = file_info_match.group(2)
        avg_score = file_info_match.group(3)

    peptide_blocks = re.split(r"Peptide No::\s*(\d+)", raw_text)[1:]

    for i in range(0, len(peptide_blocks), 2):
        peptide_no = peptide_blocks[i].strip()
        block = peptide_blocks[i + 1]

        seq_match = re.search(r"\s([A-Z]+)\s", block)
        sequence = seq_match.group(1) if seq_match else ""

        length_match = re.search(r"Length::(\d+)", block)
        length = length_match.group(1) if length_match else ""

        pred_len_match = re.search(r"Predicted length::(\d+)", block)
        pred_len = pred_len_match.group(1) if pred_len_match else ""

        residues_clean = ""
        residues_section = re.search(
            r"Predicted Residue::(.*?)(JMOL|View PDB|Predicted length::|$)",
            block,
            re.DOTALL,
        )
        if residues_section:
            residues_raw = residues_section.group(1).strip()
            residues_list = [r.strip() for r in residues_raw.split(",") if r.strip()]
            converted_residues = convert_to_three_letter_code(residues_list)
            residues_clean = "; ".join(converted_residues)

        yield {
            "protein_name": protein_name,
            "center_residue": center_residue,
            "total_score": total_score,
            "avg_score": avg_score,
            "peptide_no": peptide_no,
            "sequence": sequence,
            "length": length,
            "amino_acids": residues_clean,
            "pred_len": pred_len,
        }

def build_episearch_processed(input_patterns, output_filename=None):
    input_files = []
    for pattern in input_patterns:
        input_files.extend(glob.glob(pattern))

    if not input_files:
        raise FileNotFoundError("No files found matching the provided patterns.")

    first_base_name = os.path.basename(sorted(input_files)[0])
    first_stem, _ = os.path.splitext(first_base_name)
    output_filename = output_filename or f"{first_stem}_processed.txt"

    total_peptides = 0
    with open(output_filename, "w") as out_f:
        header = (
            "Protein Name\tCenter Residue\tEpiSearch Total Score\tEpiSearch Average Score\t"
            "Peptide Number\tPeptide Sequence\tLength\tAmino Acids (Predicted Residues)\tPredicted Length\n"
        )
        out_f.write(header)

        for filepath in sorted(input_files):
            base_name = os.path.basename(filepath)
            protein_name, _ = os.path.splitext(base_name)

            with open(filepath, "r", errors="ignore") as in_f:
                content = in_f.read()

            is_first_peptide_in_file = True
            for data_row in process_file_content(content, protein_name):
                total_peptides += 1
                if is_first_peptide_in_file:
                    line = (
                        f"{data_row['protein_name']}\t{data_row['center_residue']}\t"
                        f"{data_row['total_score']}\t{data_row['avg_score']}\t"
                        f"{data_row['peptide_no']}\t{data_row['sequence']}\t"
                        f"{data_row['length']}\t{data_row['amino_acids']}\t{data_row['pred_len']}\n"
                    )
                    is_first_peptide_in_file = False
                else:
                    line = (
                        f"\t\t\t\t"
                        f"{data_row['peptide_no']}\t{data_row['sequence']}\t"
                        f"{data_row['length']}\t{data_row['amino_acids']}\t{data_row['pred_len']}\n"
                    )
                out_f.write(line)

    print(f"Combined results saved to '{output_filename}'")
    print(f"Total peptides found: {total_peptides}")
    return output_filename

def normalize_antigen_name(antigen_name):
    if not antigen_name:
        return antigen_name
    normalized = antigen_name.strip()
    if normalized.endswith("_episearch"):
        normalized = normalized[:-len("_episearch")]
    return normalized

def parse_episearch_file(episearch_file):
    antigen_data = {}
    peptide_data = {}

    with open(episearch_file, "r") as f:
        lines = f.readlines()

    current_antigen = None
    current_center_residue = None
    current_total_score = None
    current_avg_score = None

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith("Protein Name"):
            continue

        parts = [p.strip() for p in line.split("\t")]

        if len(parts) == 9:
            current_antigen = normalize_antigen_name(parts[0])
            current_center_residue = parts[1]
            current_total_score = parts[2]
            current_avg_score = parts[3]
            peptide_seq = parts[5]
            predicted_residues = parts[7]
            predicted_length = parts[8]

            if current_antigen not in antigen_data:
                antigen_data[current_antigen] = {
                    "center_residue": current_center_residue,
                    "total_score": current_total_score,
                    "avg_score": current_avg_score,
                }

            if peptide_seq:
                peptide_data[peptide_seq] = {
                    "predicted_residues": predicted_residues,
                    "predicted_length": predicted_length,
                }

        elif len(parts) == 5 and current_antigen is not None:
            peptide_seq = parts[1]
            predicted_residues = parts[3]
            predicted_length = parts[4]

            if peptide_seq:
                peptide_data[peptide_seq] = {
                    "predicted_residues": predicted_residues,
                    "predicted_length": predicted_length,
                }

    return antigen_data, peptide_data

def parse_pepsurf_file(pepsurf_file):
    data = []
    with open(pepsurf_file, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            data.append(row)
    return data

def format_column_names(pepsurf_headers):
    formatted_headers = []
    name_mapping = {
        "Amino_acids": "PepSurf Predicted Residues",
        "Minimal_peptide": "Minimal Peptides",
        "Coverage_%": "Coverage_%",
        "Number_of_the_Cluster": "Number of the Cluster",
        "PepSurf_Score": "PepSurf Score",
        "Peptide_ID": "Peptide ID",
        "Peptide_Sequence": "Peptide Sequence",
        "Path_Residues": "Path Residues",
        "Coverage_n": "Coverage_n",
    }

    for header in pepsurf_headers:
        if header in name_mapping:
            formatted_headers.append(name_mapping[header])
        else:
            formatted_headers.append(header.replace("_", " "))

    return formatted_headers

def calculate_peptide_length(peptide_sequence):
    if not peptide_sequence or not isinstance(peptide_sequence, str):
        return 0

    peptide_sequence = peptide_sequence.strip()
    if " " in peptide_sequence:
        return len([aa for aa in peptide_sequence.split() if aa])
    return len(peptide_sequence)

def combine_data(pepsurf_file, episearch_file, output_file=None):
    episearch_antigens, episearch_peptides = parse_episearch_file(episearch_file)
    pepsurf_data = parse_pepsurf_file(pepsurf_file)

    pepsurf_headers = list(pepsurf_data[0].keys()) if pepsurf_data else []
    formatted_pepsurf_headers = format_column_names(pepsurf_headers)

    pepsurf_headers_up_to_seq = []
    for header, formatted_header in zip(pepsurf_headers, formatted_pepsurf_headers):
        pepsurf_headers_up_to_seq.append(formatted_header)
        if header == "Peptide_Sequence":
            break

    pepsurf_headers_up_to_seq.append("Length")

    pepsurf_headers_remaining = []
    found_path_residues = False
    for header, formatted_header in zip(pepsurf_headers, formatted_pepsurf_headers):
        if header == "Path_Residues":
            found_path_residues = True
        if found_path_residues:
            pepsurf_headers_remaining.append(formatted_header)

    episearch_headers = [
        "EpiSearch Center Residue",
        "EpiSearch Total Score",
        "EpiSearch Average Score",
        "Peptide Sequence (EpiSearch)",
        "EpiSearch Predicted Residues",
        "Predicted Length",
    ]

    output_headers = pepsurf_headers_up_to_seq + pepsurf_headers_remaining + episearch_headers

    output_data = []
    antigen_center_info_added = set()
    single_episearch_info = None
    if len(episearch_antigens) == 1:
        single_episearch_info = next(iter(episearch_antigens.values()))

    last_antigen = None
    for pepsurf_row in pepsurf_data:
        raw_antigen = pepsurf_row.get("Antigen", "")
        if raw_antigen:
            last_antigen = normalize_antigen_name(raw_antigen)
        antigen = last_antigen or ""
        peptide_seq = pepsurf_row.get("Peptide_Sequence", "").strip()

        output_row = {}
        for header, formatted_header in zip(pepsurf_headers, formatted_pepsurf_headers):
            output_row[formatted_header] = pepsurf_row.get(header, "")
            if header == "Peptide_Sequence":
                break

        output_row["Length"] = calculate_peptide_length(peptide_seq)

        found_path_residues = False
        for header, formatted_header in zip(pepsurf_headers, formatted_pepsurf_headers):
            if header == "Path_Residues":
                found_path_residues = True
            if found_path_residues:
                output_row[formatted_header] = pepsurf_row.get(header, "")

        antigen_key = f"{antigen}_header"
        antigen_info = None
        if antigen in episearch_antigens:
            antigen_info = episearch_antigens[antigen]
        elif single_episearch_info is not None:
            antigen_info = single_episearch_info

        if antigen_info is not None and antigen_key not in antigen_center_info_added:
            output_row["EpiSearch Center Residue"] = antigen_info.get("center_residue", "")
            output_row["EpiSearch Total Score"] = antigen_info.get("total_score", "")
            output_row["EpiSearch Average Score"] = antigen_info.get("avg_score", "")
            antigen_center_info_added.add(antigen_key)
        else:
            output_row["EpiSearch Center Residue"] = ""
            output_row["EpiSearch Total Score"] = ""
            output_row["EpiSearch Average Score"] = ""

        if peptide_seq in episearch_peptides:
            pep_info = episearch_peptides[peptide_seq]
            output_row["Peptide Sequence (EpiSearch)"] = peptide_seq
            output_row["EpiSearch Predicted Residues"] = pep_info.get("predicted_residues", "")
            output_row["Predicted Length"] = pep_info.get("predicted_length", "")
        else:
            output_row["Peptide Sequence (EpiSearch)"] = ""
            output_row["EpiSearch Predicted Residues"] = ""
            output_row["Predicted Length"] = ""

        output_data.append(output_row)

    output_file = output_file or "combined_pepsurf_episearch_results.csv"
    with open(output_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=output_headers)
        writer.writeheader()
        writer.writerows(output_data)

    print(f"Combined data written to: {output_file}")
    print(f"Total rows: {len(output_data)}")
    print(f"Total peptides matched: {sum(1 for row in output_data if row.get('Peptide Sequence (EpiSearch)'))}")

    return output_file

def parse_residues(residue_string):
    residues = {}
    if pd.isna(residue_string):
        return residues

    for item in str(residue_string).split(";"):
        item = item.strip()
        if not item:
            continue
        match = re.search(r"([A-Z]{3})(\d+)", item)
        if match:
            aa = match.group(1)
            pos = int(match.group(2))
            residues[pos] = f"{aa}{pos}"
    return residues

def residues_to_string(res_dict):
    if not res_dict:
        return ""
    return "; ".join([res_dict[k] for k in sorted(res_dict)])

def residues_to_pymol(res_dict):
    if not res_dict:
        return ""
    resi_numbers = "+".join(str(k) for k in sorted(res_dict))
    return f"select epitope, resi {resi_numbers}\ncolor yellow, epitope"

def pepsurf_episearch_union(input_csv):
    df = pd.read_csv(input_csv)

    df["Epitope EpiSearch"] = ""
    df["Epitope PepSurf U EpiSearch"] = ""
    df["Union Length"] = ""
    df["Pymol coloring command"] = ""

    current_antigen = None
    antigen_start_idx = None
    episearch_residues = {}

    for idx, row in df.iterrows():
        if pd.notna(row.get("Antigen")) and str(row.get("Antigen")).strip() != "":
            if current_antigen is not None:
                epi_str = residues_to_string(episearch_residues)
                df.at[antigen_start_idx, "Epitope EpiSearch"] = epi_str

                epi_clean = parse_residues(epi_str)
                pepsurf_res = parse_residues(
                    df.at[antigen_start_idx, "PepSurf Predicted Residues"]
                )

                union_res = pepsurf_res.copy()
                union_res.update(epi_clean)

                df.at[antigen_start_idx, "Epitope PepSurf U EpiSearch"] = residues_to_string(union_res)
                df.at[antigen_start_idx, "Union Length"] = str(len(union_res))
                df.at[antigen_start_idx, "Pymol coloring command"] = residues_to_pymol(union_res)

            current_antigen = row.get("Antigen")
            antigen_start_idx = idx
            episearch_residues = {}

        epi_res = parse_residues(row.get("EpiSearch Predicted Residues"))
        episearch_residues.update(epi_res)

    if current_antigen is not None:
        epi_str = residues_to_string(episearch_residues)
        df.at[antigen_start_idx, "Epitope EpiSearch"] = epi_str

        epi_clean = parse_residues(epi_str)
        pepsurf_res = parse_residues(
            df.at[antigen_start_idx, "PepSurf Predicted Residues"]
        )

        union_res = pepsurf_res.copy()
        union_res.update(epi_clean)

        df.at[antigen_start_idx, "Epitope PepSurf U EpiSearch"] = residues_to_string(union_res)
        df.at[antigen_start_idx, "Union Length"] = str(len(union_res))
        df.at[antigen_start_idx, "Pymol coloring command"] = residues_to_pymol(union_res)

    base, _ = os.path.splitext(input_csv)
    output_csv = f"{base}_union.csv"
    df.to_csv(output_csv, index=False)

    print(f"Saved: {output_csv}")
    return output_csv

In [ ]:
# Step 7: Process EpiSearch file and download processed output
episearch_file = expected_episearch
if not os.path.exists(episearch_file):
    if episearch_upload:
        episearch_file = next(iter(episearch_upload.keys()))
    if not os.path.exists(episearch_file):
        raise FileNotFoundError(f"EpiSearch file not found: {expected_episearch}")

episearch_processed = build_episearch_processed(
    [episearch_file],
    output_filename=f"{antigen_name}_episearch_processed.txt",
)

from google.colab import files
files.download(episearch_processed)

**Expected output:** Download of `<antigen_name>_episearch_processed.txt`.

## PepSurf_G ∪ EpiSearch
After running Step 7, you can determine the epitope by running the Step 8 to get the union of PepSurf and EpiSearch results.

In [ ]:
# Step 8: Combine PepSurf + EpiSearch and build union table
combined_csv = combine_data(
    step3_csv,
    episearch_processed,
    output_file=f"{antigen_name}_combined_pepsurf_episearch.csv",
)
union_csv = pepsurf_episearch_union(combined_csv)

print("Generated files:")
print("-", combined_csv)
print("-", union_csv)

**Expected output:** Creates `<antigen_name>_combined_pepsurf_episearch.csv` and `<antigen_name>_combined_pepsurf_episearch_union.csv`.

In [ ]:
# Step 9: Download final union outputs (CSV + Excel)
from google.colab import files
import pandas as pd

if not os.path.exists(union_csv):
    raise FileNotFoundError(f"Missing output: {union_csv}")

union_xlsx = union_csv.replace(".csv", ".xlsx")
pd.read_csv(union_csv).to_excel(union_xlsx, index=False)

files.download(union_csv)
files.download(union_xlsx)

**Expected output:** Downloads the union CSV and Excel files.

## Three-dimensional structure visualization (optional)
Upload a PDB file and render the epitope residues from the PepSurf_G ∪ EpiSearch.

Non-epitope residues are shown as light gray cartoon; epitope residues are yellow spheres.

In [ ]:
#@title 3D visualization (click to expand)
# Step 10: Upload PDB and visualize epitope residues
from google.colab import files
import pandas as pd
import re

try:
    import py3Dmol
except ImportError:
    import sys
    !{sys.executable} -m pip -q install py3Dmol
    import py3Dmol

if not os.path.exists(union_csv):
    raise FileNotFoundError(f"Missing union CSV: {union_csv}")

df_union = pd.read_csv(union_csv)
command_col = "Pymol coloring command"
if command_col not in df_union.columns:
    raise ValueError(f"Column not found in union CSV: {command_col}")

command_text = ""
for val in df_union[command_col].dropna().astype(str):
    if "select epitope" in val:
        command_text = val
        break
if not command_text:
    raise ValueError("No epitope selection command found in union CSV")

resi_match = re.search(r"resi\s+([0-9+]+)", command_text)
if not resi_match:
    raise ValueError("Could not parse residue list from command")
residue_numbers = resi_match.group(1).split("+")

print("Upload a .pdb file for visualization")
pdb_upload = files.upload()
pdb_path = next(iter(pdb_upload.keys()))

with open(pdb_path, "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=900, height=600)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "#D3D3D3"}})
view.setStyle({"resi": residue_numbers}, {"sphere": {"color": "yellow"}})
view.zoomTo()
view.show()

**Expected output:** Interactive 3D viewer with yellow spheres on epitope residues.

## Troubleshooting
- Missing file errors: check the exact filenames and re-upload the required files.
- Empty EpiSearch processed file: confirm the copied EpiSearch solution includes the full block from "Solution No" to "Predicted length".
- No epitope selection command: ensure the union CSV was created and includes the `Pymol coloring command` column.
- 3D view looks wrong: make sure PDB residue numbering matches the union residue numbers.
- Download does not start: allow pop-ups or re-run the download cell.

 Technical correspondence: alexsander.moraes@usp.br 
 
 Subject [PepSurf_G ∪ EpiSearch Collab Notebook]